In [1]:
import ee
import geemap
 
# Initialize Earth Engine
ee.Initialize()
 
# Load the Hansen Global Forest Change dataset
dataset = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')
 
# Define West Kalimantan boundary
west_kalimantan = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(
    ee.Filter.eq('ADM1_NAME', 'Kalimantan Barat')
)
 
# Create an interactive map
Map = geemap.Map()
Map.centerObject(west_kalimantan, 8)
 
# Add West Kalimantan boundary
Map.addLayer(west_kalimantan, {'color': 'blue'}, 'West Kalimantan Boundary', False)
 
# Create forest loss mask for 2000-2024
forest_loss = dataset.select(['loss']).clip(west_kalimantan)
 
# Visualize forest loss
forest_loss_vis_param = {
    'palette': ['red'],
    'min': 0,
    'max': 1
}
Map.addLayer(forest_loss.selfMask(), forest_loss_vis_param, 'Forest Loss 2000-2024')
 
# Optional: Show loss by year
tree_loss_year = dataset.select(['lossyear']).clip(west_kalimantan)
tree_loss_vis_param = {
    'min': 0,
    'max': 24,
    'palette': ['yellow', 'red']
}
Map.addLayer(tree_loss_year.selfMask(), tree_loss_vis_param, 'Forest Loss by Year', False)
 
# Display the map
Map

Map(center=[-0.08939312490516992, 111.11798765080799], controls=(WidgetControl(options=['position', 'transpare…

In [2]:
# Export forest loss to Google Drive
task1 = ee.batch.Export.image.toDrive(
    image=forest_loss,
    description='west_kalimantan_forest_loss_2000_2024',
    folder='GEE_Exports',
    fileNamePrefix='west_kalimantan_forest_loss',
    region=west_kalimantan.geometry(),
    scale=30,
    maxPixels=1e13,
    crs='EPSG:4326'
)
task1.start()
 
# Export loss year data
task2 = ee.batch.Export.image.toDrive(
    image=tree_loss_year,
    description='west_kalimantan_forest_loss_year_2000_2024',
    folder='GEE_Exports',
    fileNamePrefix='west_kalimantan_forest_loss_year',
    region=west_kalimantan.geometry(),
    scale=30,
    maxPixels=1e13,
    crs='EPSG:4326'
)
task2.start()
 
print("Export tasks started. Check your Google Earth Engine Tasks tab.")
print("Task 1:", task1.status())
print("Task 2:", task2.status())

Export tasks started. Check your Google Earth Engine Tasks tab.
Task 1: {'state': 'READY', 'description': 'west_kalimantan_forest_loss_2000_2024', 'priority': 100, 'creation_timestamp_ms': 1759848794142, 'update_timestamp_ms': 1759848794142, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'PJZXRDR4SBE3B4LNSLDTRP3A', 'name': 'projects/351618454440/operations/PJZXRDR4SBE3B4LNSLDTRP3A'}
Task 2: {'state': 'READY', 'description': 'west_kalimantan_forest_loss_year_2000_2024', 'priority': 100, 'creation_timestamp_ms': 1759848795034, 'update_timestamp_ms': 1759848795034, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'DP3Q6CR6FGZA5AQM2IOOI6XO', 'name': 'projects/351618454440/operations/DP3Q6CR6FGZA5AQM2IOOI6XO'}
